In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

#Data download
!wget -N 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

--2026-10-11 01:01:03--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.004s  

Last-modified header missing -- time-stamps turned off.
2026-10-11 01:01:04 (67.8 MB/s) - ‘course_lead_scoring_2026.csv’ saved [278746/278746]



,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [2]:
#DATA PREPARATION

#Define categorical and numerical columns
categorical = ["industry","location","lead_source","employment_status"]
numerical = ["annual_income","number_of_courses_viewed","interaction_count","lead_score"]

#Check missing vaalues
print(df.isnull().sum())

#Iterate over each column and fill null values with NA (for string) or 0.0 for float.
for col in df.columns:
    typ = df[col].dtype
    if str(typ) == "str":
        df[col] = df[col].fillna("NA")
    elif typ == np.float64:
        df[col] = df[col].fillna(0.0)
        
    #print(col)
    #print(df[col].unique()[:5]) #returns unique value in column (first five)
    #print(df[col].nunique()) #returns numnber of unique values
    #print(df[col].count()) #returns count of non-null values
    #print()

print(df.isnull().sum())
    


lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64
lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


In [3]:
#Question 1 - What is the most frequent observation (mode) for the column industry
print(df["industry"].mode().tolist())

#Answer - technology

['technology']


In [4]:
#Question 2 - Calculate Correlation across features
print(df["interaction_count"].corr(df["lead_score"]))
print(df["number_of_courses_viewed"].corr(df["lead_score"]))
print(df["number_of_courses_viewed"].corr(df["interaction_count"]))
print(df["annual_income"].corr(df["interaction_count"]))

#Answer - 1st option interaction_count x lead_score

0.9157457980418677
0.7572042392104887
0.7216090038937009
0.12284235135955954


In [5]:
#Data Split

from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

#Separate converted column (as target)
y_train = df_train['converted']
y_val = df_val['converted']
y_test = df_test['converted']
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [6]:
#Question 3 - Calculate mutual information score between categorial values and converted (target)



from sklearn.metrics import mutual_info_score

def mutual_info_churn_score(series):
    return mutual_info_score(series, y_train)
 
mi = round(df_train[categorical].apply(mutual_info_churn_score),2)
mi.sort_values(ascending=False)

#Answer - lead_source

lead_source          0.03
employment_status    0.02
location             0.00
industry             0.00
dtype: float64

In [7]:
#Qustion 4 - Logistic Regression

#One hot enconding to transform categorical columns in numerical
dicts = df[categorical + numerical].to_dict(orient='records') #define dicitionary with columns

from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse=False) #initialize dictionary

train_dict = df_train[categorical + numerical].to_dict(orient='records') #create training X dictionary
X_train = dv.fit_transform(train_dict) # Create matrix of training with categorical columns transformed

#Do same for validation
val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [8]:
print(dv.get_feature_names_out()) #Check columns with one-hot
print(X_train.shape) #check shape

['annual_income' 'employment_status=NA' 'employment_status=employed'
 'employment_status=self_employed' 'employment_status=student'
 'employment_status=unemployed' 'industry=NA' 'industry=education'
 'industry=finance' 'industry=healthcare' 'industry=manufacturing'
 'industry=retail' 'industry=technology' 'interaction_count' 'lead_score'
 'lead_source=NA' 'lead_source=events' 'lead_source=organic_search'
 'lead_source=paid_ads' 'lead_source=referral' 'lead_source=social_media'
 'location=NA' 'location=africa' 'location=asia' 'location=europe'
 'location=north_america' 'location=south_america'
 'number_of_courses_viewed']
(3000, 28)


In [9]:
#Question 4 - continuation - Logistic Model (binary)

#Define sigmoid function to squashes any real number into the range between 0 and 1
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

#Model for logicis regression (simple - similar to regression but with sigmoid) - THIS IS FOR LEARNING ONLY WILL NOT BE USED IN THE HOMEWORK
def logistic_regression(xi):
    score = w0
    
    for j in range(len(w)):
        score = score + xi[j] * w[j]
        
    result = sigmoid(score)
    return result

#Another model from sklearn
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42) #define model

model.fit(X_train, y_train)  # Fit the model

y_pred = model.predict_proba(X_val)[:, 1] #Perform prediction with val data

prob = (y_pred >= 0.5) #Checl model probability
acc = (y_val == prob).mean() #Calculate accuracy
print(round(acc,2)) #check prediction accuracy



0.64


In [10]:
#Question 5 - Same but removing some features
ColumnToRemove = ["lead_source","number_of_courses_viewed","interaction_count"]
#Removing requried columns

for Column in ColumnToRemove:
    dicts1 = df[categorical + numerical].to_dict(orient='records') #define dicitionary with columns
    dv1 = DictVectorizer(sparse=False) #initialize dictionary
    train_dict1 = ((df_train[categorical + numerical]).drop(Column, axis=1)).to_dict(orient='records') #create training X dictionary without given feature
    X_train1 = dv.fit_transform(train_dict1) # Create matrix of training with categorical columns transformed
    val_dict1 = (df_val[categorical + numerical].drop(Column, axis=1)).to_dict(orient='records')
    X_val1 = dv.transform(val_dict1)
    
    #Model training
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42) #define model
    model.fit(X_train1, y_train)  # Fit the model
    y_pred = model.predict_proba(X_val1)[:, 1] #Perform prediction with val data
    acc1 = (y_val == (y_pred >= 0.5)).mean() #Calculate accuracy
    print(Column)
    print(acc-acc1)
    print()
    
#Answer - number_of_courses_viewed


lead_source
0.0030000000000000027

number_of_courses_viewed
0.0020000000000000018

interaction_count
0.04400000000000004



In [11]:
#Question 6 - Same as 4 but with different C
CList = [0.000001, 0.00001, 0.0001, 0.001]

for cValue in CList:
    model = LogisticRegression(solver='liblinear', C=cValue, max_iter=1000, random_state=42) #define model
    model.fit(X_train, y_train)  # Fit the model
    y_pred = model.predict_proba(X_val)[:, 1] #Perform prediction with val data
    acc2 = (y_val == (y_pred >= 0.5)).mean() #Calculate accuracy
    print(cValue)
    print(round(acc2,3))
    print()

#Answer - 0.000001

1e-06
0.598

1e-05
0.598

0.0001
0.613

0.001
0.645

